# 5. Statistical Testing

## Introduction

In this notebook, statistical tests will be performed to evaluate the significance of the predictors and examine potential multicollinearity.

## Objectives

- Evaluate the statistical significance of the regression coefficients.
- Test whether the TV–Radio interaction contributes meaningfully to the model.
- Examine the contribution of the nonlinear TV transformation.
- Assess multicollinearity using Variance Inflation Factor (VIF).
- Compare conventional and HC3 robust standard errors.

## Statistical Questions

1. Are the nonlinear TV and Radio predictors statistically significant?
2. Does the TV–Radio interaction have a significant relationship with Sales?
3. Is multicollinearity present among the predictors?
4. Do conclusions about coefficient significance change when robust standard errors are used?

## Approach

The analysis will use the training dataset (160 observations). The test dataset will remain untouched until final model evaluation.

The Nonlinear TV + Interaction Model will be the primary model investigated, with the Reduced Additive Model serving as a reference.

## 1. Import Libraries and Load Data

The training dataset will be loaded using the reusable utility functions created earlier.

Only the training data will be used for statistical testing. The test dataset will remain untouched until the final evaluation stage.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_and_split

sns.set_theme(style="whitegrid")

df_train, df_test = load_and_split(
    path=PROJECT_ROOT / "Dataset" / "Advertising.csv"
)

df = df_train.copy()

print("Training data shape:", df_train.shape)

Training data shape: (160, 4)


## 2. Prepare Features for Statistical Testing

The selected candidate model from Notebook 4 uses a square-root transformation of TV advertising expenditure and an interaction between TV and Radio.

Before fitting the model:

- Apply the square-root transformation to TV.
- Mean-center the transformed TV variable.
- Mean-center Radio advertising expenditure.
- Create the interaction term between the centered variables.

Mean-centering makes the main-effect coefficients easier to interpret and can help reduce non-essential multicollinearity between interaction terms and their component variables.

In [2]:
df["sqrt_tv"] = np.sqrt(df["tv"])

df["sqrt_tv_c"] = (
    df["sqrt_tv"] - df["sqrt_tv"].mean()
)

df["radio_c"] = (
    df["radio"] - df["radio"].mean()
)

df["interaction"] = (
    df["sqrt_tv_c"] * df["radio_c"]
)

df.head()

,tv,radio,newspaper,sales,sqrt_tv,sqrt_tv_c,radio_c,interaction
80,116.0,7.7,23.1,11.0,10.770330,-0.799756,-15.175625,12.136804
198,177.0,9.3,6.4,12.8,13.304135,1.734049,-13.575625,-23.540794
39,43.1,26.7,35.1,10.1,6.565059,-5.005027,3.824375,-19.141100
25,62.3,12.6,18.3,9.7,7.893035,-3.677051,-10.275625,37.783999
123,224.0,2.4,15.6,11.6,14.966630,3.396543,-20.475625,-69.546351


## 3. Fit the Nonlinear Interaction Model

The Nonlinear TV + Interaction Model will be fitted using Ordinary Least Squares (OLS).

### Model Specification

Sales = β₀ + β₁(√TV_c) + β₂(Radio_c) + β₃(√TV_c × Radio_c) + ε

The model includes:

- Centered square-root transformed TV expenditure.
- Centered Radio expenditure.
- An interaction between the two predictors.

The OLS summary will be examined to assess coefficient estimates, statistical significance, confidence intervals, and overall model performance.

In [3]:
model_d = smf.ols(
    "sales ~ sqrt_tv_c * radio_c",
    data=df
).fit()

print(model_d.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.992
Model:                            OLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     6600.
Date:                Wed, 25 Nov 2026   Prob (F-statistic):          4.56e-164
Time:                        13:45:21   Log-Likelihood:                -99.372
No. Observations:                 160   AIC:                             206.7
Df Residuals:                     156   BIC:                             219.0
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            14.0561      0.03

### Model D — Statistical Findings

1. The model achieved an R-squared of 0.992 and an Adjusted R-squared of 0.992, explaining approximately 99.2% of the variation in Sales on the training dataset.

2. The transformed TV variable (`sqrt_tv_c`) is statistically significant (p < 0.001), indicating a positive association with Sales.

3. Radio expenditure (`radio_c`) is statistically significant (p < 0.001), showing a positive association with Sales.

4. The TV–Radio interaction term is statistically significant (p < 0.001), suggesting that the relationship between one advertising channel and Sales depends partly on the expenditure level of the other channel.

5. The model's condition number is 61.8. Multicollinearity will be investigated further using Variance Inflation Factor (VIF).

### Conclusion

All three predictors are statistically significant in the Nonlinear TV + Interaction Model. The results support retaining the nonlinear TV transformation and interaction term for further investigation.

However, coefficient inference will also be examined using HC3 robust standard errors, and regression assumptions will be assessed in the model diagnostics notebook.

## 4. Multicollinearity Assessment Using VIF

Variance Inflation Factor (VIF) measures how much the variance of a regression coefficient is inflated due to linear relationships among the predictors.

Higher VIF values indicate greater multicollinearity.

The VIF will be calculated for the three predictors in Model D:

- `sqrt_tv_c`
- `radio_c`
- `interaction`

The results will help determine whether multicollinearity could affect the stability and interpretation of the regression coefficients.

In [4]:

X_vif = df[
    ["sqrt_tv_c", "radio_c", "interaction"]
].copy()

vif_df = pd.DataFrame({
    "Feature": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

vif_df.round(3)

,Feature,VIF
0,sqrt_tv_c,1.007
1,radio_c,1.002
2,interaction,1.007


### VIF — Key Findings

1. The VIF values for all three predictors are close to 1.
2. `sqrt_tv_c` has a VIF of 1.007, `radio_c` has 1.002, and the interaction term has 1.007.
3. These values indicate negligible multicollinearity among the model predictors.

### Conclusion

The VIF analysis indicates that multicollinearity is not a concern in the Nonlinear TV + Interaction Model.

The mean-centered predictors and interaction term show very low linear dependence, supporting the stability of the model's coefficient estimates with respect to multicollinearity.

## 5. Robust Standard Errors Using HC3

The conventional OLS standard errors assume that the error variance is constant across observations.

If heteroscedasticity is present, conventional standard errors may be unreliable, potentially affecting hypothesis tests and confidence intervals.

To address this concern, HC3 heteroscedasticity-robust standard errors will be calculated.

The coefficient estimates remain unchanged, but the standard errors, t-statistics, p-values, and confidence intervals may differ.

In [5]:
model_d_hc3 = model_d.get_robustcov_results(
    cov_type="HC3"
)

print(model_d_hc3.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.992
Model:                            OLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     5054.
Date:                Wed, 25 Nov 2026   Prob (F-statistic):          4.13e-155
Time:                        13:45:21   Log-Likelihood:                -99.372
No. Observations:                 160   AIC:                             206.7
Df Residuals:                     156   BIC:                             219.0
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            14.0561      0.03

### HC3 Robust Standard Errors — Key Findings

1. HC3 robust standard errors were calculated to account for potential heteroscedasticity in the regression errors.

2. The regression coefficients and R-squared remained unchanged because HC3 adjusts the covariance estimates rather than the fitted model.

3. The robust standard errors increased slightly compared with conventional OLS standard errors.

4. All three predictors (`sqrt_tv_c`, `radio_c`, and their interaction) remained statistically significant, with p-values below 0.001.

### Conclusion

The statistical significance of all three predictors remains consistent after applying HC3 robust standard errors.

This provides stronger evidence that the observed coefficient relationships are not solely dependent on the conventional OLS standard-error assumptions.

## 6. Nested Model F-Test

A nested model F-test will be performed to determine whether adding the TV–Radio interaction term significantly improves the model compared with an additive specification.

### Hypotheses

- H₀: The interaction coefficient is equal to zero.
- H₁: The interaction coefficient is not equal to zero.

A significance level of 0.05 will be used.

If the p-value is below 0.05, we reject the null hypothesis and conclude that the interaction term provides a statistically significant improvement in model fit.

In [6]:
model_additive = smf.ols(
    "sales ~ sqrt_tv_c + radio_c",
    data=df
).fit()

model_interaction = smf.ols(
    "sales ~ sqrt_tv_c * radio_c",
    data=df
).fit()

f_test = model_interaction.compare_f_test(model_additive)

print("F-statistic:", f_test[0])
print("P-value:", f_test[1])
print("Degrees of freedom difference:", f_test[2])

F-statistic: 1329.665307685666
P-value: 3.036206282602508e-78
Degrees of freedom difference: 1.0


### Nested Model F-Test — Key Findings

1. The nested F-test was conducted to compare the additive model against the model containing the TV–Radio interaction term.

2. The test produced an F-statistic of 1329.665 and a p-value of 3.04 × 10⁻⁷⁸.

3. Since the p-value is below the 0.05 significance level, the null hypothesis is rejected.

4. The interaction term provides a statistically significant improvement in model fit over the additive model.

### Conclusion

The nested model F-test provides strong statistical evidence supporting the inclusion of the TV–Radio interaction term.

Combined with the significance of the transformed TV and Radio predictors, this supports retaining the Nonlinear TV + Interaction Model for further diagnostic evaluation.